# 04. Balance and rerandomization

Randomization balances covariates **on average**, but a specific draw can come out imbalanced. This notebook shows how to **measure** balance (SMD) and how rerandomization improves a bad draw before running the experiment.

## Measure balance with SMD

The standardized mean difference (SMD) compares treated and controls on each covariate. Rule of thumb: `|SMD| > 0.1` flags relevant imbalance.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.design.balance import check_balance
from skxperiments.design.crd import CRD

rng = np.random.default_rng(3)
n = 100
df = pd.DataFrame({"x1": rng.normal(size=n), "x2": rng.normal(size=n)})

crd = CRD(p=0.5, seed=3).randomize(df)
check_balance(crd)[["covariate", "smd"]]

## Rerandomization

`ReRandomizedCRD` re-draws until the Mahalanobis distance between the groups falls below a threshold. The result is a draw with smaller SMDs.

In [ ]:
from scipy.stats import chi2

from skxperiments.design.rerandomized_crd import ReRandomizedCRD

threshold = float(chi2.ppf(0.05, df=2))   # accepts the ~5% most balanced draws
rerand = ReRandomizedCRD(
    covariates=["x1", "x2"], threshold=threshold, p=0.5, seed=3, max_attempts=10000
).randomize(df)
check_balance(rerand)[["covariate", "smd"]]

## Report and plot

`BalanceReport` summarizes the SMDs and flags imbalances; `plot_balance` draws the Love plot (needs matplotlib, in the `viz` extra).

In [ ]:
from skxperiments.diagnostics import BalanceReport
from skxperiments.reporting import plot_balance

report = BalanceReport().run(rerand)
print("Imbalanced:", report.imbalanced)
ax = plot_balance(report)
ax.figure

## What we learned

- SMD measures the balance of each covariate; `|SMD| > 0.1` is a warning sign.
- Rerandomization rejects imbalanced draws **before** collecting data, improving balance without violating randomization.
- `BalanceReport` and `plot_balance` make this visible in a report.

**Next:** `05. Blocking` guarantees balance on known categorical variables.